# HI5312 — SQL JOINs with MIMIC-IV
## Interactive companion lab

This notebook accompanies the **SQL JOINs with MIMIC-IV** presentation.

### Core question
> **What does one row represent before—and after—the JOIN?**

We will work primarily in the **Codespaces terminal with `psql`**. The notebook is your work-along guide, prediction sheet, and reflection space.

### MIMIC-IV tables used
- `mimiciv_hosp.patients`
- `mimiciv_hosp.admissions`
- `mimiciv_icu.icustays`
- `mimiciv_hosp.diagnoses_icd`
- `mimiciv_hosp.d_icd_diagnoses`
- `mimiciv_hosp.labevents`
- `mimiciv_hosp.d_labitems`

MIMIC-IV Demo uses the same schema/structure as the corresponding MIMIC-IV release and contains a subset of patients for educational exploration.

## 1. Connect to PostgreSQL

Open the Codespaces terminal and run:

```bash
psql -h db -U hi5304_user -d hi5304
```

Then turn off the pager:

```sql
\pset pager off
```

Inspect the available schemas:

```sql
\dn
```

Inspect MIMIC-IV tables:

```sql
\dt mimiciv_hosp.*
\dt mimiciv_icu.*
```

### Quick check
What do you expect these identifiers to represent?

- `subject_id` = __________________
- `hadm_id` = _____________________
- `stay_id` = _____________________

## 2. Inspect before joining

Run:

```sql
SELECT *
FROM mimiciv_hosp.patients
LIMIT 5;
```

```sql
SELECT *
FROM mimiciv_hosp.admissions
LIMIT 5;
```

```sql
SELECT *
FROM mimiciv_icu.icustays
LIMIT 5;
```

### Before moving on
Complete this table:

| Table | What does one row represent? | Important identifier(s) |
|---|---|---|
| patients | | |
| admissions | | |
| icustays | | |

Do not write a JOIN until you can describe the grain of each source table.

## 3. First JOIN: patients + admissions

### Predict first

```sql
SELECT
    p.subject_id,
    p.gender,
    p.anchor_age,
    a.hadm_id,
    a.admittime,
    a.dischtime
FROM mimiciv_hosp.patients AS p
JOIN mimiciv_hosp.admissions AS a
    ON p.subject_id = a.subject_id
ORDER BY p.subject_id, a.admittime
LIMIT 30;
```

Before running:

1. Can `subject_id` appear more than once?
2. Can `hadm_id` appear more than once?
3. What should one row represent after this JOIN?

Now run the query.

## 4. Validate the first JOIN

Run:

```sql
SELECT
    COUNT(*) AS rows,
    COUNT(DISTINCT p.subject_id) AS patients,
    COUNT(DISTINCT a.hadm_id) AS admissions
FROM mimiciv_hosp.patients AS p
JOIN mimiciv_hosp.admissions AS a
    ON p.subject_id = a.subject_id;
```

### Discuss
If `rows` is greater than `patients`, is that necessarily a problem?

Why or why not?

## 5. INNER JOIN: admissions + ICU stays

```sql
SELECT
    a.subject_id,
    a.hadm_id,
    a.admission_type,
    i.stay_id,
    i.first_careunit,
    i.intime,
    i.outtime,
    i.los
FROM mimiciv_hosp.admissions AS a
INNER JOIN mimiciv_icu.icustays AS i
    ON a.hadm_id = i.hadm_id
ORDER BY a.subject_id, a.hadm_id
LIMIT 30;
```

### Predict before running
Which admissions will appear?

A. All hospital admissions  
B. Only admissions with a matching ICU stay  
C. Only admissions without an ICU stay

Run the query and verify your prediction.

## 6. LEFT JOIN: keep all admissions

Now change only the JOIN type:

```sql
SELECT
    a.subject_id,
    a.hadm_id,
    a.admission_type,
    i.stay_id,
    i.first_careunit,
    i.los
FROM mimiciv_hosp.admissions AS a
LEFT JOIN mimiciv_icu.icustays AS i
    ON a.hadm_id = i.hadm_id
ORDER BY a.subject_id, a.hadm_id
LIMIT 30;
```

### Look for NULL
What does a `NULL` `stay_id` mean here?

Write your interpretation:

____________________________________________________________

## 7. Compare INNER and LEFT JOIN counts

Run both:

```sql
SELECT COUNT(*) AS inner_rows
FROM mimiciv_hosp.admissions AS a
INNER JOIN mimiciv_icu.icustays AS i
    ON a.hadm_id = i.hadm_id;
```

```sql
SELECT COUNT(*) AS left_rows
FROM mimiciv_hosp.admissions AS a
LEFT JOIN mimiciv_icu.icustays AS i
    ON a.hadm_id = i.hadm_id;
```

### Reflection
Why are the counts different?

Which JOIN would you use for:

> “Describe only hospitalizations involving ICU care.”

Which JOIN would you use for:

> “Describe all hospitalizations and indicate whether ICU care occurred.”

In [ ]:
# Optional notebook knowledge check — Python, not SQL.
answer = input("Which JOIN keeps every row from the left table? ")
if answer.strip().lower() in {"left", "left join"}:
    print("✓ Correct — LEFT JOIN.")
else:
    print("Review INNER JOIN vs LEFT JOIN and try again.")

## 8. One-to-many: admissions + diagnoses

First inspect:

```sql
SELECT *
FROM mimiciv_hosp.diagnoses_icd
LIMIT 10;
```

Now JOIN:

```sql
SELECT
    a.subject_id,
    a.hadm_id,
    a.admittime,
    d.seq_num,
    d.icd_code,
    d.icd_version
FROM mimiciv_hosp.admissions AS a
JOIN mimiciv_hosp.diagnoses_icd AS d
    ON a.hadm_id = d.hadm_id
ORDER BY a.hadm_id, d.seq_num
LIMIT 40;
```

### Critical question
What does one row represent **now**?

It is no longer simply one hospital admission.

Write the new grain:

____________________________________________________________

## 9. See row multiplication directly

Pick admissions with multiple diagnosis rows:

```sql
SELECT
    hadm_id,
    COUNT(*) AS diagnosis_rows
FROM mimiciv_hosp.diagnoses_icd
GROUP BY hadm_id
ORDER BY diagnosis_rows DESC
LIMIT 10;
```

Choose one `hadm_id` from the result and inspect it:

```sql
SELECT *
FROM mimiciv_hosp.diagnoses_icd
WHERE hadm_id = YOUR_HADM_ID
ORDER BY seq_num;
```

### Discuss
If an admission has 8 diagnosis rows, how many rows can that admission contribute after joining to `diagnoses_icd`?

## 10. Add a dictionary table

ICD codes are more useful when we add their descriptions.

```sql
SELECT
    d.subject_id,
    d.hadm_id,
    d.seq_num,
    d.icd_code,
    d.icd_version,
    dd.long_title
FROM mimiciv_hosp.diagnoses_icd AS d
JOIN mimiciv_hosp.d_icd_diagnoses AS dd
    ON d.icd_code = dd.icd_code
   AND d.icd_version = dd.icd_version
ORDER BY d.hadm_id, d.seq_num
LIMIT 30;
```

### Why are there TWO conditions in `ON`?

```sql
d.icd_code = dd.icd_code
AND d.icd_version = dd.icd_version
```

Explain:

____________________________________________________________

## 11. Laboratory events + laboratory dictionary

Inspect the two tables:

```sql
SELECT *
FROM mimiciv_hosp.labevents
LIMIT 5;
```

```sql
SELECT *
FROM mimiciv_hosp.d_labitems
LIMIT 5;
```

Now join them:

```sql
SELECT
    l.subject_id,
    l.hadm_id,
    l.charttime,
    l.itemid,
    dl.label,
    l.valuenum,
    l.valueuom
FROM mimiciv_hosp.labevents AS l
JOIN mimiciv_hosp.d_labitems AS dl
    ON l.itemid = dl.itemid
WHERE l.valuenum IS NOT NULL
LIMIT 30;
```

### Identify the roles
- Event table = __________________
- Dictionary table = _____________
- Join key = _____________________
- One result row = _______________

## 12. Three-table JOIN: patient → admission → ICU

Build incrementally.

### Step 1

```sql
SELECT
    p.subject_id,
    p.gender,
    p.anchor_age,
    a.hadm_id,
    a.admission_type
FROM mimiciv_hosp.patients AS p
JOIN mimiciv_hosp.admissions AS a
    ON p.subject_id = a.subject_id
LIMIT 20;
```

### Step 2 — add ICU stays

```sql
SELECT
    p.subject_id,
    p.gender,
    p.anchor_age,
    a.hadm_id,
    a.admission_type,
    a.hospital_expire_flag,
    i.stay_id,
    i.first_careunit,
    i.los
FROM mimiciv_hosp.patients AS p
JOIN mimiciv_hosp.admissions AS a
    ON p.subject_id = a.subject_id
JOIN mimiciv_icu.icustays AS i
    ON a.hadm_id = i.hadm_id
LIMIT 30;
```

### After each step ask:
**What does one row represent now?**

## 13. Validate the three-table result

```sql
SELECT
    COUNT(*) AS rows,
    COUNT(DISTINCT p.subject_id) AS patients,
    COUNT(DISTINCT a.hadm_id) AS admissions,
    COUNT(DISTINCT i.stay_id) AS icu_stays
FROM mimiciv_hosp.patients AS p
JOIN mimiciv_hosp.admissions AS a
    ON p.subject_id = a.subject_id
JOIN mimiciv_icu.icustays AS i
    ON a.hadm_id = i.hadm_id;
```

### Interpretation
If:

- rows = ICU stays
- ICU stays > admissions

what might that tell you?

____________________________________________________________

## 14. A subtle LEFT JOIN trap

Compare these two queries.

### Query A

```sql
SELECT a.hadm_id, i.stay_id, i.first_careunit
FROM mimiciv_hosp.admissions AS a
LEFT JOIN mimiciv_icu.icustays AS i
    ON a.hadm_id = i.hadm_id
WHERE i.first_careunit = 'Medical Intensive Care Unit (MICU)';
```

### Query B

```sql
SELECT a.hadm_id, i.stay_id, i.first_careunit
FROM mimiciv_hosp.admissions AS a
LEFT JOIN mimiciv_icu.icustays AS i
    ON a.hadm_id = i.hadm_id
   AND i.first_careunit = 'Medical Intensive Care Unit (MICU)';
```

### Predict
Will these queries return the same population?

Why can a `WHERE` condition on the right-hand table change the practical behavior of a LEFT JOIN?

## 15. Debugging JOINs

For each problem, identify the risk.

### A
```sql
... JOIN diagnoses_icd d
ON a.subject_id = d.subject_id
```

Would `subject_id` alone necessarily keep diagnoses within the correct hospitalization?

### B
```sql
... JOIN d_icd_diagnoses dd
ON d.icd_code = dd.icd_code
```

What additional field should be matched?

### C
Your query returned 10 times as many rows after adding `labevents`.

Is that automatically an error?

### D
Your query runs successfully.

Does that prove the JOIN is analytically correct?

## 16. Build an analytic extract

Goal: **one row per ICU stay**, with patient and hospital context.

```sql
SELECT
    p.subject_id,
    p.gender,
    p.anchor_age,
    a.hadm_id,
    a.admission_type,
    a.admittime,
    a.dischtime,
    a.hospital_expire_flag,
    i.stay_id,
    i.first_careunit,
    i.intime,
    i.outtime,
    i.los
FROM mimiciv_hosp.patients AS p
JOIN mimiciv_hosp.admissions AS a
    ON p.subject_id = a.subject_id
JOIN mimiciv_icu.icustays AS i
    ON a.hadm_id = i.hadm_id
ORDER BY p.subject_id, a.admittime;
```

### Before exporting
Check:

```sql
SELECT COUNT(*) FROM (...your query without ORDER BY...) AS x;
```

Also ask:
- Is `stay_id` unique?
- Are there unexpected NULL values?
- Is ICU LOS plausible?
- Does one row still represent one ICU stay?

# 17. Interactive challenge

Write a query that returns:

- `subject_id`
- `hadm_id`
- `admission_type`
- `stay_id`
- `first_careunit`
- `los`

for **only hospital admissions that had an ICU stay**.

Before running it, answer:

1. Which two tables?
2. Which JOIN key?
3. INNER or LEFT JOIN?
4. What will one row represent?

<details>
<summary>Show solution after you try it</summary>

```sql
SELECT
    a.subject_id,
    a.hadm_id,
    a.admission_type,
    i.stay_id,
    i.first_careunit,
    i.los
FROM mimiciv_hosp.admissions AS a
INNER JOIN mimiciv_icu.icustays AS i
    ON a.hadm_id = i.hadm_id
ORDER BY a.subject_id, a.hadm_id;
```

</details>

## 18. Exit ticket

In your own words:

### A. What is a JOIN?
____________________________________________________________

### B. What is the difference between INNER JOIN and LEFT JOIN?
____________________________________________________________

### C. Why can JOINs increase the number of rows?
____________________________________________________________

### D. What should you ask before and after every JOIN?
____________________________________________________________

### E. Name the three key MIMIC-IV identifiers introduced today.
1. __________________
2. __________________
3. __________________

## 19. Close the session

```sql
\q
```

### Key takeaway

> **A technically successful JOIN can still produce an analytically incorrect dataset.**

Always define the grain, choose the correct identifier, predict the relationship, and validate the result.

### Sources
- PhysioNet MIMIC-IV Clinical Database Demo v2.2
- PhysioNet MIMIC-IV documentation